# Modeling: Telecom Customer Churn

**Objective:** Build and compare models that predict whether a customer will churn.

**Steps in this notebook**
1. Load the cleaned data
2. Feature engineering
3. Train/test split
4. Preprocessing pipeline
5. Baseline and model comparison (next stage)

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/telco_clean.csv")
df.shape

(7043, 31)

## 1. Feature Engineering

New features can help the model see patterns that are not obvious in the raw columns.

- `num_services`: how many of the 10 services a customer uses (more services may mean a stronger tie to the company)
- `tenure_group`: groups tenure into 0-12, 13-24, 25-48 and 49-72 months, because risk drops sharply after the first year
- `has_support_addon`: 1 if the customer has online security or premium tech support (both were linked to lower churn in the EDA)

**Data leakage rule:** every new feature uses only information that would be known at prediction time. None of them uses the target (`churn`).

In [4]:
service_cols = ["phone_service", "multiple_lines", "online_security", "online_backup",
                "device_protection_plan", "premium_tech_support", "streaming_tv",
                "streaming_movies", "streaming_music", "unlimited_data"]

df["num_services"] = df[service_cols].sum(axis=1)

df["tenure_group"] = pd.cut(df["tenure_in_months"],
                            bins=[0, 12, 24, 48, 72],
                            labels=["0-12m", "13-24m", "25-48m", "49-72m"])

df["has_support_addon"] = ((df["online_security"] == 1) |
                           (df["premium_tech_support"] == 1)).astype(int)

In [5]:
print(df["tenure_group"].isnull().sum())   # should be 0
print(df.groupby("tenure_group", observed=True)["churn"].mean().round(3))

0
tenure_group
0-12m     0.474
13-24m    0.287
25-48m    0.204
49-72m    0.095
Name: churn, dtype: float64


## 2. Train/Test Split

We split the data 80% for training and 20% for testing.

- `stratify=y` keeps the churn ratio (about 26.5%) the same in both sets.
- `random_state=42` makes the split reproducible.
- **The test set stays untouched until the final evaluation.** We never tune or choose models using it.

In [10]:
from sklearn.model_selection import train_test_split

X = df.drop(columns="churn")
y = df["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(round(y_train.mean(), 3), round(y_test.mean(), 3))

(5634, 33) (1409, 33)
0.265 0.265


## 3. Preprocessing Pipeline

Models need numbers, so we:
- **Scale** numeric columns (`StandardScaler`) so large values such as `total_charges` don't dominate small ones
- **One-hot encode** text categories (`OneHotEncoder`), turning each category into its own 0/1 column
- **Pass through** columns that are already 0/1

Putting this in a `ColumnTransformer` means the same steps are applied identically in training, testing and the web app. The encoder and scaler are **fitted on training data only**, which prevents leakage from the test set.

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_features = ["age", "number_of_dependents", "population", "number_of_referrals",
                    "tenure_in_months", "avg_monthly_long_distance_charges",
                    "avg_monthly_gb_download", "monthly_charge", "total_charges",
                    "total_refunds", "total_extra_data_charges",
                    "total_long_distance_charges", "num_services"]

categorical_features = ["gender", "offer", "internet_type", "contract",
                        "payment_method", "tenure_group"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ],
    remainder="passthrough",   # keeps the 0/1 columns as they are
)

In [11]:
X_train_t = preprocessor.fit_transform(X_train)   # fit on train ONLY
X_test_t = preprocessor.transform(X_test)         # only transform the test set

print(X_train_t.shape, X_test_t.shape)
print(len(preprocessor.get_feature_names_out()))

(5634, 49) (1409, 49)
49


### Feature Engineering Summary
- Added 3 features: `num_services`, `tenure_group`, `has_support_addon`
- Train set: 5,634 customers; test set: 1,409 customers (churn rate 26.5% in both)
- Preprocessing: scaling for numeric columns, one-hot encoding for categorical columns, pass-through for binary columns
- Next: train a baseline and compare Logistic Regression, Random Forest and XGBoost using 5-fold cross-validation